# Crowd-navigation policy comparison
This notebook compares persisted runs across every built-in CrowdSimPlus scenario. Expensive policies are not silently rerun: set `RUN_MISSING=True` deliberately after solver setup is complete. Use the same case IDs, human count, simulator policy, and randomization setting for paired comparisons.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / 'experiments/crowd_navigation/navigation.py').exists())
EXP = ROOT / 'experiments/crowd_navigation'
sys.path.insert(0, str(EXP))
from navigation import BUILTIN_SCENARIOS, METHODS, dependency_report, load_results, run_navigation_suite
OUTPUT = EXP / 'outputs/policy_benchmark'
pd.DataFrame([dependency_report(method) for method in METHODS])

## Benchmark matrix
Three humans preserve fidelity with the released SICNav-JMID simulation checkpoint. Run higher-density stress tests separately for policies that naturally support variable crowd sizes.

In [ ]:
RUN_MISSING = False
SELECTED_METHODS = list(METHODS)
SELECTED_SCENARIOS = list(BUILTIN_SCENARIOS)
HUMAN_COUNT = 3
CASES = range(5)
if RUN_MISSING:
    for method in SELECTED_METHODS:
        for scenario in SELECTED_SCENARIOS:
            expected = OUTPUT / f'{method}__{scenario}__n{HUMAN_COUNT}.csv'
            if expected.exists():
                print('already exists:', expected.name)
                continue
            run_navigation_suite(method, cases=CASES, scenario=scenario, human_count=HUMAN_COUNT, starts_moving=10, device='auto', output_dir=OUTPUT)

## Aggregate outcomes
Collision rate is the fraction of episodes containing at least one collision, not the fraction of simulator timesteps in collision.

In [ ]:
raw = load_results(OUTPUT)
if raw.empty:
    raise RuntimeError('No benchmark CSVs found. Run a policy notebook or set RUN_MISSING=True above.')
summary = (raw.assign(collision_episode=raw.collision_steps.gt(0), wall_collision_episode=raw.wall_collision_steps.gt(0))
    .groupby(['method','scenario','human_count'], as_index=False)
    .agg(episodes=('test_case','count'), success_rate=('success','mean'), collision_rate=('collision_episode','mean'), wall_collision_rate=('wall_collision_episode','mean'), timeout_rate=('timeout','mean'), mean_navigation_time_s=('navigation_time_s','mean'), mean_clearance_m=('minimum_clearance_m','mean'), mean_path_efficiency=('path_efficiency','mean'), mean_policy_time_ms=('mean_policy_time_ms','mean')))
display(summary.sort_values(['scenario','method']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, metric, title in zip(axes, ['success_rate','collision_rate','mean_navigation_time_s'], ['Success rate','Episodes with collision','Navigation time (s)']):
    summary.pivot(index='scenario', columns='method', values=metric).plot.bar(ax=ax)
    ax.set_title(title); ax.set_xlabel(''); ax.grid(axis='y', alpha=.2); ax.legend(fontsize=8)
plt.show()

## Interpretation guardrails
A fair result needs repeated seeds and paired cases. ORCA humans favor ORCA-based robot models; repeat selected scenarios with SFM humans to measure simulator-model mismatch. Compare safety, freezing, efficiency, smoothness, and compute together rather than ranking policies from success rate alone.